In [ ]:
#1. Retrieval: gives results from vector store based on query
# def retrieve: Retrieve relevant documents from the vector store in form of a list of dictionaries 
# query is tranformed into an embedding and then used to search the vector store for similar documents

import sys
sys.path.insert(0, '../')

from typing import List, Dict, Any
from src.embedding_manager import EmbeddingManager
from src.vector_store import FaissVectorStore

class RAGRetriever:
    def __init__(self, vector_store: FaissVectorStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        
        print(f"Retrieving documents for query: '{query}' with top_k={top_k} and score_threshold={score_threshold}")
        
        try:
            # Generate embedding for the query
            query_embeddings = self.embedding_manager.generate_embeddings([query])
            query_embedding = query_embeddings[0]
            
            # Search the vector store - returns list of tuples (metadata, distance)
            results = self.vector_store.search(query_embedding, top_k=top_k)
            retrieved_docs = []
            
            for rank, (metadata, distance) in enumerate(results, 1):
                similarity_score = distance
                
                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        'id': rank,
                        'content': metadata.get('content', ''),
                        'source': metadata.get('source', ''),
                        'similarity_score': float(similarity_score),
                        'metadata': metadata,
                        'rank': rank
                    })
            
            if retrieved_docs:
                print(f"Retrieved {len(retrieved_docs)} documents above the score threshold of {score_threshold}")
            else:
                print(f"No documents retrieved above the score threshold of {score_threshold}")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            import traceback
            traceback.print_exc()
            return []

In [ ]:
#2. Initialize vector store and embedding manager (load from disk)

try:
    embedding_manager = EmbeddingManager()
    print("EmbeddingManager initialized successfully")
except Exception as e:
    print(f"Failed to initialize EmbeddingManager: {e}")
    embedding_manager = None

try:
    vector_store = FaissVectorStore(embedding_dim=embedding_manager.get_embedding_dimension())
    print(f"Vector store loaded successfully with {len(vector_store.id_to_metadata)} chunks")
except Exception as e:
    print(f"Failed to initialize Vector Store: {e}")
    vector_store = None

In [ ]:
# 3. Initialize RAGRetriever with instances from ingestion.ipynb

rag_retriever = RAGRetriever(vector_store=vector_store, embedding_manager=embedding_manager)

query = "What is the economic contribution of the Great Barrier Reef?"
results = rag_retriever.retrieve(query=query, top_k=5, score_threshold=0.0)

for doc in results:
    
    print(f"\nRank {doc['rank']}: {doc['similarity_score']:.4f}")
    print(f"Source: {doc['source']}")
    print(f"Content preview: {doc['content'][:200]}...")


In [ ]:
# 3b. Debug: Check raw similarity scores

rag_retriever = RAGRetriever(vector_store=vector_store, embedding_manager=embedding_manager)

query = "What is the economic contribution of the Great Barrier Reef?"

# Get results with NO threshold filtering
results_no_filter = rag_retriever.retrieve(query=query, top_k=5, score_threshold=0.0)

print("Raw similarity scores (without threshold filter):")
for doc in results_no_filter:
    print(f"  Rank {doc['rank']}: {doc['similarity_score']:.4f}")


In [ ]:
#4. LLM integration with Ollama

from langchain_ollama import OllamaLLM
import os
from dotenv import load_dotenv
load_dotenv()

#temperature defines how "creative" the model's answers will be --> low value for solid and factual answers
llm = OllamaLLM(model="llama3", temperature=0.2)


In [ ]:
#5. Simple RAG function that combines retrieval and generation

def rag_answer(query: str, retriever: RAGRetriever, llm: OllamaLLM, top_k: int = 5, score_threshold: float = 0.3):

    results = retriever.retrieve(query=query, top_k=top_k, score_threshold=score_threshold)
    
    if not results:
        return "No relevant documents found to answer the question."

    context = "\n".join([doc['content'] for doc in results])
    
    prompt = f"""Use the following context to answer the question concisely and factually.

        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response = llm.invoke(prompt)
    return response

In [ ]:
#6. Test the RAG function
answer=rag_answer("how many full time jobs does the GBR offer?", rag_retriever, llm, score_threshold=0.3)
print(answer)

In [ ]:
#7. enhance : returns answer but also source, confidence score and optionally the context used for answering

def rag_answer_enhanced(query: str, retriever: RAGRetriever, llm: OllamaLLM, top_k: int = 5, min_score=0.2, return_context=False):
    results = retriever.retrieve(query=query, top_k=top_k, score_threshold=min_score)
    
    if not results:
        return {
            'answer': "No relevant documents found to answer the question.",
            'sources': [],
            'confidence': 0.0
        }
    
    # Prepare context and sources
    context = "\n".join([doc['content'] for doc in results])
    sources = [
        {
            'source': doc['source'],
            'score': round(doc['similarity_score'], 3),
            'preview': doc['content'][:200] + "..."
        }
        for doc in results
    ]
    
    confidence = max(doc['similarity_score'] for doc in results)


    # generate answer
    prompt = f"""You are a precise research assistant. Use the provided context to answer the question.
        Rely on the clear facts directly mentioned in the context. Be short and concise. 
        If the answer cannot be found, say 'I cannot find the answer.'

CONTEXT:
{context}

QUESTION: {query}

ANSWER:"""
    
    answer = llm.invoke(prompt)
    
    output = {
        'answer': answer,
        'sources': sources,
        'confidence': round(float(confidence), 3)
    }
    
    if return_context:
        output['context'] = context
    
    return output


# Test the enhanced function
result = rag_answer_enhanced("how many full time jobs does the GBR offer?", rag_retriever, llm, top_k=3, min_score=0.3, return_context=True)
print("Answer:", result['answer'])
print("\nSources:", result['sources'])
print("Confidence Score:", result['confidence'])


Retrieving documents for query: 'how many full time jobs does the GBR offer?' with top_k=2 and score_threshold=0.3


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.16it/s]


Retrieved 2 documents above the score threshold of 0.3
Answer: According to the context, about 66,000 persons are employed (full-time equivalent basis) in the GBRCA.

Sources: [{'source': 'Access Economics 2007 Economic contribution of Great Barrier Reef Marine Park 2005-2006 Kopie.pdf', 'score': 0.512, 'preview': 'For employment (full time equivalent basis), about 66,000 persons. Tourism dominates these contributions: For value-added and gross product, tourism’s share is about 84%-87%. For employment, tourism’s...'}, {'source': 'Access Economics 2007 Economic contribution of Great Barrier Reef Marine Park 2005-2006 Kopie.pdf', 'score': 0.478, 'preview': 'GAP, GSP and GDP are fractions of the appropriate GVP. 26 TABLE 5.2.1: DIRECT CONTRIBUTIONS OF SELECTED ACTIVITIES TO THE GBRCA,-05 &-06-05-05-05-06-06-06 Direct contribution Direct Direct Direct Dire...'}]
Confidence Score: 0.512
